# Influencer Campaign ROI Analysis

## Project Overview
This notebook analyzes influencer marketing campaign results across creators, platforms, and campaigns. We focus on engagement, conversions, customer acquisition cost (CAC), and return on spend (ROAS). Synthetic data models typical DTC brand influencer programs.

## Learning Objectives
- Calculate influencer marketing ROI metrics: CPE, CAC, ROAS.
- Compare influencer tiers (nano, micro, macro) on cost efficiency.
- Identify which platforms and content formats drive actual conversions vs vanity metrics.
- Build a data framework for evaluating influencer partnerships.

## Problem Statement
A DTC brand runs influencer campaigns across multiple creators and platforms. They need to determine which influencer partnerships generate real returns and which are consuming budget without proportional impact.

## Why This Project Matters
Influencer marketing spend is growing rapidly but measurement remains weak. Brands that rigorously track ROI per creator and platform can reallocate budget from low-performing partnerships to high-impact ones.

## Dataset Overview
Synthetic dataset of ~500 influencer campaign activations with fields: influencer name, tier, platform, campaign, content type, followers, impressions, engagements, link clicks, conversions, spend, and revenue.

## Dataset Source and License Notes
Synthetic data generated in-notebook. No external dependencies.

## Environment Setup

In [ ]:
import importlib, subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## Imports

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')
np.random.seed(42)

## Configuration / Constants

In [ ]:
TIERS = ['Nano (1K-10K)', 'Micro (10K-50K)', 'Macro (50K-500K)', 'Mega (500K+)']
PLATFORMS = ['Instagram', 'TikTok', 'YouTube', 'Twitter/X']
CAMPAIGNS = ['Spring Launch', 'Summer Sale', 'Back to School', 'Holiday', 'New Product', 'Brand Awareness']
CONTENT_TYPES = ['Post', 'Story', 'Reel', 'Video', 'Unboxing']
N_ACTIVATIONS = 500

## Dataset Generation

In [ ]:
rng = np.random.default_rng(42)

tier_config = {
    'Nano (1K-10K)':    {'followers': (1000, 10000),   'cpm': 8,   'eng_rate': 0.06, 'cvr': 0.025},
    'Micro (10K-50K)':  {'followers': (10000, 50000),  'cpm': 15,  'eng_rate': 0.04, 'cvr': 0.018},
    'Macro (50K-500K)': {'followers': (50000, 500000), 'cpm': 25,  'eng_rate': 0.025, 'cvr': 0.010},
    'Mega (500K+)':     {'followers': (500000, 5000000), 'cpm': 40, 'eng_rate': 0.015, 'cvr': 0.006},
}

influencer_names = [f'Creator_{i:03d}' for i in range(80)]

records = []
for _ in range(N_ACTIVATIONS):
    tier = rng.choice(TIERS, p=[0.30, 0.35, 0.25, 0.10])
    cfg = tier_config[tier]
    platform = rng.choice(PLATFORMS, p=[0.35, 0.30, 0.20, 0.15])
    campaign = rng.choice(CAMPAIGNS)
    content = rng.choice(CONTENT_TYPES, p=[0.25, 0.15, 0.25, 0.20, 0.15])
    creator = rng.choice(influencer_names[:20] if tier == 'Nano (1K-10K)' else
                          influencer_names[20:45] if tier == 'Micro (10K-50K)' else
                          influencer_names[45:65] if tier == 'Macro (50K-500K)' else
                          influencer_names[65:])

    followers = int(rng.uniform(*cfg['followers']))
    impressions = int(followers * rng.uniform(0.15, 0.60))
    eng_rate = cfg['eng_rate'] * rng.lognormal(0, 0.3)
    engagements = int(impressions * eng_rate)
    clicks = int(engagements * rng.uniform(0.08, 0.25))
    cvr = cfg['cvr'] * rng.lognormal(0, 0.4)
    conversions = int(clicks * cvr)
    spend = impressions / 1000 * cfg['cpm'] * rng.uniform(0.8, 1.3)
    aov = rng.uniform(35, 120)
    revenue = conversions * aov

    date = pd.Timestamp('2025-01-01') + pd.Timedelta(days=int(rng.integers(0, 365)))

    records.append({
        'Date': date, 'Creator': creator, 'Tier': tier, 'Platform': platform,
        'Campaign': campaign, 'Content Type': content, 'Followers': followers,
        'Impressions': impressions, 'Engagements': engagements,
        'Link Clicks': clicks, 'Conversions': conversions,
        'Spend ($)': round(spend, 2), 'Revenue ($)': round(revenue, 2),
    })

df = pd.DataFrame(records)
df['Engagement Rate %'] = np.where(df['Impressions'] > 0, df['Engagements'] / df['Impressions'] * 100, 0)
df['CTR %'] = np.where(df['Impressions'] > 0, df['Link Clicks'] / df['Impressions'] * 100, 0)
df['CVR %'] = np.where(df['Link Clicks'] > 0, df['Conversions'] / df['Link Clicks'] * 100, 0)
df['CPE ($)'] = np.where(df['Engagements'] > 0, df['Spend ($)'] / df['Engagements'], np.nan)
df['CAC ($)'] = np.where(df['Conversions'] > 0, df['Spend ($)'] / df['Conversions'], np.nan)
df['ROAS'] = np.where(df['Spend ($)'] > 0, df['Revenue ($)'] / df['Spend ($)'], 0)

print(f'Dataset shape: {df.shape}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Rows: {len(df)}, Columns: {len(df.columns)}')
print(f'Nulls:\n{df.isna().sum().to_string()}')
print(f'Tiers: {sorted(df["Tier"].unique())}')
print(f'Platforms: {sorted(df["Platform"].unique())}')
print(f'Unique creators: {df["Creator"].nunique()}')
print(f'Campaigns: {sorted(df["Campaign"].unique())}')

## Exploratory Data Analysis

### Overall Campaign Metrics

In [ ]:
total_spend = df['Spend ($)'].sum()
total_revenue = df['Revenue ($)'].sum()
total_conversions = df['Conversions'].sum()

print('Overall Influencer Program Metrics:')
print(f'  Total activations: {len(df):,}')
print(f'  Total spend: ${total_spend:,.0f}')
print(f'  Total revenue: ${total_revenue:,.0f}')
print(f'  Overall ROAS: {total_revenue/total_spend:.2f}x')
print(f'  Total conversions: {total_conversions:,}')
print(f'  Avg CAC: ${total_spend/max(total_conversions,1):,.2f}')
print(f'  Total impressions: {df["Impressions"].sum():,.0f}')

### Performance by Influencer Tier
Compare nano, micro, macro, and mega influencers on efficiency metrics.

In [ ]:
tier_perf = df.groupby('Tier').agg(
    Activations=('ROAS', 'count'),
    Total_Spend=('Spend ($)', 'sum'),
    Total_Revenue=('Revenue ($)', 'sum'),
    Total_Conversions=('Conversions', 'sum'),
    Avg_Engagement_Rate=('Engagement Rate %', 'mean'),
    Avg_ROAS=('ROAS', 'mean'),
    Median_CAC=('CAC ($)', 'median'),
).round(2)
tier_perf['Overall_ROAS'] = (tier_perf['Total_Revenue'] / tier_perf['Total_Spend']).round(2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(range(len(tier_perf)), tier_perf['Overall_ROAS'], color='seagreen', edgecolor='black')
axes[0].set_xticks(range(len(tier_perf)))
axes[0].set_xticklabels(tier_perf.index, rotation=20, ha='right')
axes[0].set_title('ROAS by Tier')
axes[0].set_ylabel('ROAS')
axes[0].axhline(1.0, color='red', linestyle='--', label='Breakeven')
axes[0].legend()

axes[1].bar(range(len(tier_perf)), tier_perf['Avg_Engagement_Rate'], color='steelblue', edgecolor='black')
axes[1].set_xticks(range(len(tier_perf)))
axes[1].set_xticklabels(tier_perf.index, rotation=20, ha='right')
axes[1].set_title('Avg Engagement Rate by Tier')
axes[1].set_ylabel('ER %')

axes[2].bar(range(len(tier_perf)), tier_perf['Median_CAC'].fillna(0), color='coral', edgecolor='black')
axes[2].set_xticks(range(len(tier_perf)))
axes[2].set_xticklabels(tier_perf.index, rotation=20, ha='right')
axes[2].set_title('Median CAC by Tier')
axes[2].set_ylabel('CAC ($)')

plt.tight_layout()
plt.show()
print(tier_perf.to_string())

### Performance by Platform

In [ ]:
plat_perf = df.groupby('Platform').agg(
    Activations=('ROAS', 'count'),
    Total_Spend=('Spend ($)', 'sum'),
    Total_Revenue=('Revenue ($)', 'sum'),
    Avg_ROAS=('ROAS', 'mean'),
    Avg_ER=('Engagement Rate %', 'mean'),
    Avg_CTR=('CTR %', 'mean'),
).round(2)
plat_perf['Overall_ROAS'] = (plat_perf['Total_Revenue'] / plat_perf['Total_Spend']).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(plat_perf.index, plat_perf['Overall_ROAS'], color='teal', edgecolor='black')
axes[0].set_title('ROAS by Platform')
axes[0].axhline(1.0, color='red', linestyle='--')

axes[1].bar(plat_perf.index, plat_perf['Avg_ER'], color='mediumpurple', edgecolor='black')
axes[1].set_title('Avg Engagement Rate by Platform')

plt.tight_layout()
plt.show()
print(plat_perf.to_string())

### Performance by Campaign

In [ ]:
camp_perf = df.groupby('Campaign').agg(
    Total_Spend=('Spend ($)', 'sum'),
    Total_Revenue=('Revenue ($)', 'sum'),
    Conversions=('Conversions', 'sum'),
    Avg_ROAS=('ROAS', 'mean'),
).round(2)
camp_perf['Overall_ROAS'] = (camp_perf['Total_Revenue'] / camp_perf['Total_Spend']).round(2)
camp_perf = camp_perf.sort_values('Overall_ROAS', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['seagreen' if r >= 1 else 'coral' for r in camp_perf['Overall_ROAS']]
ax.barh(camp_perf.index, camp_perf['Overall_ROAS'], color=colors, edgecolor='black')
ax.axvline(1.0, color='red', linestyle='--', label='Breakeven')
ax.set_xlabel('ROAS')
ax.set_title('Campaign ROAS')
ax.legend()
plt.tight_layout()
plt.show()
print(camp_perf.to_string())

### Content Type Analysis

In [ ]:
content_perf = df.groupby('Content Type').agg(
    Avg_ER=('Engagement Rate %', 'mean'),
    Avg_CTR=('CTR %', 'mean'),
    Avg_ROAS=('ROAS', 'mean'),
).sort_values('Avg_ROAS', ascending=False).round(2)

print('Content Type Performance:')
print(content_perf.to_string())

### Top and Bottom Creators
Identify the highest and lowest ROI creators with at least 3 activations.

In [ ]:
creator_perf = df.groupby('Creator').agg(
    Activations=('ROAS', 'count'),
    Total_Spend=('Spend ($)', 'sum'),
    Total_Revenue=('Revenue ($)', 'sum'),
    Avg_ROAS=('ROAS', 'mean'),
).round(2)
creator_perf['Overall_ROAS'] = (creator_perf['Total_Revenue'] / creator_perf['Total_Spend']).round(2)

qualified = creator_perf[creator_perf['Activations'] >= 3].sort_values('Overall_ROAS', ascending=False)
print(f'Creators with 3+ activations: {len(qualified)}')
print(f'\nTop 5 Creators by ROAS:')
print(qualified.head(5).to_string())
print(f'\nBottom 5 Creators by ROAS:')
print(qualified.tail(5).to_string())

### Tier × Platform Heatmap

In [ ]:
cross_roas = df.pivot_table(values='ROAS', index='Tier', columns='Platform', aggfunc='mean')
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(cross_roas, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax, linewidths=0.5, center=1.0)
ax.set_title('Avg ROAS — Tier × Platform')
plt.tight_layout()
plt.show()

## Executive Summary

In [ ]:
insights = [
    f'Total program spend: ${total_spend:,.0f} → Revenue: ${total_revenue:,.0f} (ROAS: {total_revenue/total_spend:.2f}x)',
    f'Best tier by ROAS: {tier_perf["Overall_ROAS"].idxmax()} ({tier_perf["Overall_ROAS"].max():.2f}x)',
    f'Nano/Micro influencers typically deliver higher engagement rates and better CAC',
    f'Best platform: {plat_perf["Overall_ROAS"].idxmax()} ({plat_perf.loc[plat_perf["Overall_ROAS"].idxmax(), "Overall_ROAS"]:.2f}x ROAS)',
    f'Top campaign: {camp_perf.index[0]} ({camp_perf["Overall_ROAS"].iloc[0]:.2f}x ROAS)',
    f'{len(qualified[qualified["Overall_ROAS"] < 1])} of {len(qualified)} qualified creators are below breakeven',
]
for item in insights:
    print(f'  * {item}')

## Limitations
- Synthetic data — real influencer data has attribution challenges (last-click bias, organic lift).
- Revenue attribution assumes all conversions within a window belong to the influencer.
- No multi-touch attribution — the same customer may have seen multiple creators.
- Follower counts and engagement rates change over time; this is a snapshot.

## Recommendations
- Shift budget toward nano/micro tier for higher ROAS if brand awareness is not the goal.
- Sunset creators consistently below breakeven after 3+ activations.
- Test platform-specific content formats (Reels on IG, short videos on TikTok).
- Build an incrementality test framework to measure true lift vs organic baseline.

## Common Mistakes
- Evaluating influencers on engagement alone without tracking conversions.
- Comparing creators with different follower counts without normalizing.
- Ignoring CAC — high ROAS with tiny volume may not be worth the management overhead.
- Not distinguishing awareness campaigns from performance campaigns in ROI analysis.

## Mini Challenge
1. Calculate the incremental ROAS if you cut all creators with ROAS < 0.5 and redistribute their budget to top performers.
2. Build an "efficiency frontier" scatter plot: ROAS vs total spend per creator.
3. Find which content type × platform combination maximizes conversions per dollar.

## Final Summary / Key Takeaways
Influencer ROI analysis must go beyond engagement rates to track actual conversions, CAC, and ROAS. Smaller creators often deliver better cost efficiency, while mega influencers provide reach. The right strategy blends both tiers based on campaign objectives — but always with measurable performance tracking.